# 案例4：电力系统小干扰稳定分析

## 计算原理

小干扰稳定分析研究系统在小扰动下的动态行为，通过线性化状态方程和特征值分析判断稳定性。

**经典模型线性化：**

$$\begin{bmatrix} \Delta\dot{\delta} \ \Delta\dot{\omega} \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ -\frac{K_s}{M} & -\frac{D}{M} \end{bmatrix} \begin{bmatrix} \Delta\delta \ \Delta\omega \end{bmatrix}$$

其中同步转矩系数 $K_s = \frac{E'V_\infty}{X_\Sigma}\cos\delta_0$

**特征方程：** $Ms^2 + Ds + K_s = 0$

**特征值：** $s = \frac{-D \pm \sqrt{D^2 - 4MK_s}}{2M}$

**阻尼比：** $\zeta = \frac{-\text{Re}(\lambda)}{|\lambda|}$

**振荡频率：** $f = \frac{|\text{Im}(\lambda)|}{2\pi}$ Hz

**稳定判据：** 所有特征值实部 < 0，且阻尼比 ζ > 0

**参考教材：** 李光琦《电力系统暂态分析》第四章

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from psa4teaching.stability import analyze_single_machine_infinite_bus

## 1. 基本分析：不同阻尼系数下的特征值变化

In [ ]:
# 基本参数
E_prime = 1.2
V_inf = 1.0
X_total = 0.5
H = 5.0
delta_0 = np.radians(30)
Pm = 0.8

# 不同阻尼系数
D_values = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]

print('=== 不同阻尼系数下的特征值分析 ===')
print(f'系统参数: E\'={E_prime}, V∞={V_inf}, X={X_total}, H={H}s, δ₀={np.degrees(delta_0)}°\n')
print(f'{"阻尼D":>8} {"特征值λ₁":>20} {"特征值λ₂":>20} {"阻尼比ζ":>10} {"频率(Hz)":>10} {"稳定":>6}')
print('-' * 80)

zetas = []
freqs = []
real_parts = []

for D in D_values:
    result = analyze_single_machine_infinite_bus(
        E_prime, V_inf, X_total, H, D, delta_0, Pm
    )
    ev = result.eigenvalues
    stable = result.stable
    
    # 取有虚部的特征值
    complex_ev = [e for e in ev if abs(e.imag) > 1e-6]
    real_ev = [e for e in ev if abs(e.imag) <= 1e-6]
    
    ev_strs = []
    for e in ev:
        if abs(e.imag) > 1e-6:
            ev_strs.append(f'{e.real:.4f}+j{e.imag:.4f}')
        else:
            ev_strs.append(f'{e.real:.4f}')
    
    zeta = result.damping_ratios[0] if len(result.damping_ratios) > 0 else 0
    freq = result.frequencies[0] if len(result.frequencies) > 0 else 0
    zetas.append(zeta)
    freqs.append(freq)
    real_parts.append(ev[0].real)
    
    ev1 = ev_strs[0] if len(ev_strs) > 0 else ''
    ev2 = ev_strs[1] if len(ev_strs) > 1 else ''
    print(f'{D:>8.1f} {ev1:>20} {ev2:>20} {zeta:>10.4f} {freq:>10.3f} {"✓" if stable else "✗":>6}')

In [ ]:
# 特征值轨迹（根轨迹图）
plt.figure(figsize=(12, 5))

# 左图：特征值轨迹
plt.subplot(1, 2, 1)
for i, D in enumerate(D_values):
    result = analyze_single_machine_infinite_bus(E_prime, V_inf, X_total, H, D, delta_0, Pm)
    for ev in result.eigenvalues:
        color = 'green' if ev.real < 0 else 'red'
        marker = 'o'
        plt.plot(ev.real, ev.imag, marker, color=color, markersize=10, label=f'D={D}')
        if i < len(result.eigenvalues) - 1:
            plt.plot([result.eigenvalues[0].real, result.eigenvalues[1].real],
                     [result.eigenvalues[0].imag, result.eigenvalues[1].imag],
                     '-', color=color, alpha=0.3)

plt.axvline(x=0, color='red', linestyle='-', linewidth=2, alpha=0.5)
plt.xlabel('实部 (Re)', fontsize=12)
plt.ylabel('虚部 (Im)', fontsize=12)
plt.title('特征值分布（根轨迹）', fontsize=14)
plt.legend(fontsize=9, ncol=2)
plt.grid(True, alpha=0.3)

# 右图：阻尼比和频率
plt.subplot(1, 2, 2)
D_fine = np.linspace(0.01, 5, 100)
zetas_fine = []
freqs_fine = []
for D in D_fine:
    result = analyze_single_machine_infinite_bus(E_prime, V_inf, X_total, H, D, delta_0, Pm)
    zetas_fine.append(result.damping_ratios[0])
    freqs_fine.append(result.frequencies[0])

color1 = 'steelblue'
ax1 = plt.gca()
ax1.plot(D_fine, zetas_fine, '-', color=color1, linewidth=2, label='阻尼比 ζ')
ax1.set_xlabel('阻尼系数 D', fontsize=12)
ax1.set_ylabel('阻尼比 ζ', fontsize=12, color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'coral'
ax2.plot(D_fine, freqs_fine, '--', color=color2, linewidth=2, label='振荡频率')
ax2.set_ylabel('振荡频率 (Hz)', fontsize=12, color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('阻尼比与振荡频率随D的变化', fontsize=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('smallsignal_eigenvalue.png', dpi=150)
plt.show()

## 2. 不同运行功角的影响

In [ ]:
# 不同初始功角
delta_values = [10, 20, 30, 45, 60, 70, 80]
D_fixed = 0.1  # 固定小阻尼

print(f'=== 不同初始功角下的特征值分析（D={D_fixed}）===')
print(f'{"功角(°)":>10} {"特征值λ₁":>22} {"阻尼比ζ":>10} {"频率(Hz)":>10} {"稳定":>6}')
print('-' * 70)

zeta_delta = []
freq_delta = []
real_delta = []

for deg in delta_values:
    d_rad = np.radians(deg)
    result = analyze_single_machine_infinite_bus(E_prime, V_inf, X_total, H, D_fixed, d_rad, Pm)
    
    ev = result.eigenvalues[0]
    stable = result.stable
    zeta = result.damping_ratios[0]
    freq = result.frequencies[0]
    
    zeta_delta.append(zeta)
    freq_delta.append(freq)
    real_delta.append(ev.real)
    
    ev_str = f'{ev.real:.4f}+j{ev.imag:.4f}' if abs(ev.imag) > 1e-6 else f'{ev.real:.4f}'
    status = '✓' if stable else '✗'
    print(f'{deg:>10.0f} {ev_str:>22} {zeta:>10.4f} {freq:>10.3f} {status:>6}')

In [ ]:
# 功角影响可视化
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(delta_values, real_delta, 'b-o', linewidth=2, markersize=8)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='稳定边界')
plt.xlabel('初始功角 δ₀ (°)', fontsize=12)
plt.ylabel('特征值实部 Re(λ)', fontsize=12)
plt.title('特征值实部随功角的变化', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(delta_values, zeta_delta, 'g-o', linewidth=2, markersize=8, label='阻尼比 ζ')
ax2 = plt.gca().twinx()
ax2.plot(delta_values, freq_delta, 'r--s', linewidth=2, markersize=8, label='振荡频率')
plt.xlabel('初始功角 δ₀ (°)', fontsize=12)
plt.ylabel('阻尼比 ζ', fontsize=12, color='green')
ax2.set_ylabel('振荡频率 (Hz)', fontsize=12, color='red')
plt.title('阻尼比与频率随功角的变化', fontsize=14)
lines1, labels1 = plt.gca().get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.gca().legend(lines1 + lines2, labels1 + labels2, loc='best')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('smallsignal_delta.png', dpi=150)
plt.show()

## 3. 详细模型 vs 经典模型

In [ ]:
# 经典模型
result_classic = analyze_single_machine_infinite_bus(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    H=5.0, D=0.1, delta_0=np.radians(30), Pm=0.8,
    detailed=False
)

# 详细模型
result_detail = analyze_single_machine_infinite_bus(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    H=5.0, D=0.1, delta_0=np.radians(30), Pm=0.8,
    detailed=True,
    Xd=1.8, Xd_prime=0.3, Xq=1.7,
    Td0_prime=8.0, Ka=50.0, Te=0.3
)

print('=== 经典模型 vs 详细模型 ===')
print(f'\n经典模型（2阶）：')
print(f'  特征值: {result_classic.eigenvalues}')
print(f'  阻尼比: {result_classic.damping_ratios}')
print(f'  振荡频率: {result_classic.frequencies} Hz')
print(f'  稳定: {"是" if result_classic.stable else "否"}')

print(f'\n详细模型（4阶）：')
print(f'  特征值: ')
for i, ev in enumerate(result_detail.eigenvalues):
    zeta = result_detail.damping_ratios[i]
    freq = result_detail.frequencies[i]
    if freq > 0:
        print(f'    λ{i+1} = {ev.real:.4f} + j{ev.imag:.4f} (ζ={zeta:.4f}, f={freq:.3f}Hz)')
    else:
        print(f'    λ{i+1} = {ev.real:.4f} (非振荡模式)')
print(f'  稳定: {"是" if result_detail.stable else "否"}')

In [ ]:
# 特征值对比可视化
plt.figure(figsize=(12, 5))

# 经典模型
plt.subplot(1, 2, 1)
for ev in result_classic.eigenvalues:
    color = 'green' if ev.real < 0 else 'red'
    plt.plot(ev.real, ev.imag, 'bo', color=color, markersize=15)
    plt.annotate(f'  ({ev.real:.3f}, {ev.imag:.3f})', xy=(ev.real, ev.imag), fontsize=9)
    plt.plot(ev.real, ev.imag, 'bo', color=color, markersize=15)

plt.axvline(x=0, color='red', linestyle='-', linewidth=2, alpha=0.5)
plt.xlabel('实部 (Re)', fontsize=12)
plt.ylabel('虚部 (Im)', fontsize=12)
plt.title('经典模型特征值', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

# 详细模型
plt.subplot(1, 2, 2)
for ev in result_detail.eigenvalues:
    color = 'green' if ev.real < 0 else 'red'
    marker = 'o' if abs(ev.imag) > 1e-6 else 's'
    plt.plot(ev.real, ev.imag, marker, color=color, markersize=15)
    if abs(ev.imag) > 1e-6:
        plt.annotate(f'  ({ev.real:.3f}, {ev.imag:.3f})', xy=(ev.real, ev.imag), fontsize=8)
    else:
        plt.annotate(f'  ({ev.real:.3f})', xy=(ev.real, ev.imag), fontsize=8)

plt.axvline(x=0, color='red', linestyle='-', linewidth=2, alpha=0.5)
plt.xlabel('实部 (Re)', fontsize=12)
plt.ylabel('虚部 (Im)', fontsize=12)
plt.title('详细模型特征值', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.savefig('smallsignal_model_comparison.png', dpi=150)
plt.show()

## 4. 惯性时间常数的影响

In [ ]:
# 不同惯性常数
H_values = [2.0, 3.0, 5.0, 8.0, 10.0]
D_fixed = 0.1

print(f'=== 不同惯性常数下的振荡特性 ===')
print(f'{"H(s)":>8} {"阻尼比ζ":>10} {"振荡频率(Hz)":>14} {"振荡周期(s)":>14}')
print('-' * 50)

for H_val in H_values:
    result = analyze_single_machine_infinite_bus(
        E_prime, V_inf, X_total, H_val, D_fixed, delta_0, Pm
    )
    zeta = result.damping_ratios[0]
    freq = result.frequencies[0]
    period = 1.0 / freq if freq > 0 else float('inf')
    print(f'{H_val:>8.1f} {zeta:>10.4f} {freq:>14.3f} {period:>14.3f}')

In [ ]:
# 惯性常数影响可视化
H_fine = np.linspace(2, 15, 100)
zetas_H = []
freqs_H = []
for H_val in H_fine:
    result = analyze_single_machine_infinite_bus(E_prime, V_inf, X_total, H_val, D_fixed, delta_0, Pm)
    zetas_H.append(result.damping_ratios[0])
    freqs_H.append(result.frequencies[0])

plt.figure(figsize=(10, 6))
color1 = 'steelblue'
ax1 = plt.gca()
ax1.plot(H_fine, zetas_H, '-', color=color1, linewidth=2)
ax1.set_xlabel('惯性时间常数 H (s)', fontsize=12)
ax1.set_ylabel('阻尼比 ζ', fontsize=12, color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'coral'
ax2.plot(H_fine, freqs_H, '--', color=color2, linewidth=2)
ax2.set_ylabel('振荡频率 (Hz)', fontsize=12, color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('阻尼比与振荡频率随惯性常数H的变化', fontsize=14)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('smallsignal_inertia.png', dpi=150)
plt.show()

## 结果分析

1. **特征值物理意义：**
   - 实部 < 0：系统稳定（振荡衰减）
   - 实部 > 0：系统不稳定（振荡发散）
   - 虚部 ≠ 0：振荡模式，对应的频率为 f = |Im(λ)| / 2π
   - 虚部 = 0：非振荡模式（纯指数衰减或发散）

2. **阻尼系数D的影响：**
   - D=0时，特征值实部为零（无阻尼等幅振荡）
   - D增大，阻尼比增大，振荡更快衰减
   - 当D超过临界阻尼时，变为非振荡模式

3. **功角δ₀的影响：**
   - δ₀增大，同步转矩系数Ks = (E'V∞/XΣ)cos(δ₀)减小
   - 当δ₀接近90°时，Ks趋近于0，系统失去稳定
   - 实际运行中，δ₀通常不超过40°~50°

4. **惯性常数H的影响：**
   - H增大，振荡频率降低（系统响应变慢）
   - H增大，阻尼比减小（系统更容易振荡）
   - 工程上要求ζ > 0.05~0.1，以确保足够的阻尼

5. **经典模型 vs 详细模型：**
   - 经典模型仅有2个特征值（一对共轭复根）
   - 详细模型有4个或更多特征值，包含励磁系统等附加模式
   - 励磁系统可能引入额外的振荡模式（如低频振荡）